In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function

In [6]:
# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key = os.getenv("ROUTER_API"),
    model="gpt-oss-120b:free", 
    temperature=0.7
)

In [7]:
class Sentiment(BaseModel):
    '''Define o sentimento e o idioma da mensagem enviada'''
    sentimento: str = Field(description="Sentimento do texto, Deve ser 'pos', 'neg' ou 'nt' para não definido")
    lingua: str = Field(description="Língua ue o texto foi escrito (deve estar no charset UTF-)")

tool_sentimento = convert_to_openai_function(Sentiment)
tool_sentimento

{'name': 'Sentiment',
 'description': 'Define o sentimento e o idioma da mensagem enviada',
 'parameters': {'type': 'object',
  'properties': {'sentimento': {'type': 'string'},
   'lingua': {'type': 'string'}},
  'required': ['sentimento', 'lingua']}}

In [8]:
texto = "Eu odeio pizza nordestina"

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Analise o texto e categorize-o conforme as instruções"),
    ("user", "{input}")
])

chain = prompt | chat.bind_tools([tool_sentimento])

chain.invoke({"input": texto})

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-b61247944698aae7', 'function': {'arguments': '{\n  "lingua": "pt",\n  "sentimento": "negative"\n}', 'name': 'Sentiment'}, 'type': 'function', 'index': 0}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 151, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 64, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_name': 'gpt-oss-120b:free', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-6d68c5f6-b48c-4458-abfe-cb2ca0e6b3c8-0', tool_calls=[{'name': 'Sentiment', 'args': {'lingua': 'pt', 'sentime

### Categorização de Texto

In [10]:
solicitacoes = [
    "Meu computador está travando toda vez que tento abrir um programa. O que devo fazer?",
    "Não consio acessar minha conta. A senha não está funcionando",
    "O meu laptop não está ligando. Acho que é um problema no cabo de energia",
    "Não consigo ver meus aruivos no aplicativo. Existe alguma solução",
    "Meu cachorro está doente",
]

In [11]:
from enum import Enum

class SetorEnum(str, Enum):
    suporte_hardware = "suporte_hardware"
    suporte_software = "suporte_software"
    suporte_conta = "suporte_conta"
    outros = "outros"

class DirecionaSuporte(BaseModel):
    """Direciona a solicitação de suporte para o setor responsável"""
    setor: SetorEnum

tool_direciona = convert_to_openai_function(DirecionaSuporte)
tool_direciona

{'name': 'DirecionaSuporte',
 'description': 'Direciona a solicitação de suporte para o setor responsável',
 'parameters': {'type': 'object', 'properties': {}, 'required': ['setor']}}

In [14]:
system_message = '''Pense com cuidado ao categorizar o texto conforme as instruções
Questões relacionadas a problemas de hardware, como falha no computador, laptop ou
outros dispositivos ffísicos devem ser direcionados para "suporte_hardware".
Questões relacionadas a problemas com software, como instalação, erros ao abrir programas, etc.,
devem ser direcionadas para "suporte_software".
Problemas relacionados ao acesso da conta, como recuperação de senha, problemas de login,
devem ser direcionados para "suporte_conta".
Mensagens ue não se encaixem nessas categorias devem ser direcionadas para "outros'''

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", "{input}")
])

chain = prompt | chat.bind_tools([tool_direciona]) 

solicitacao = solicitacoes[3]

resposta = chain.invoke({"input": solicitacao})
print(resposta.content)

**Categoria:** suporte_software
